# StreamSense — TFX Training Pipeline + Kubeflow (Python 3.10, locked install)

This runs the **TFX training pipeline end-to-end** on Colab. The trick that finally fixes the
30-minute install hang: we install from a **fully-pinned lock file** (`requirements-tfx-lock.txt`,
generated with `uv`), so pip has nothing to backtrack on. We also drop full `kfp` (it conflicted
with TFX on protobuf) and use `kfp-pipeline-spec` for the Kubeflow compile.

**What this is:** the reproducible *training* pipeline — validate → transform → train → evaluate →
push — separate from serving.

> ⚠️ Run cells **in order, one at a time**. Cell 1 installs conda and **restarts the runtime on
> purpose** ("session crashed" is expected) — continue from Cell 2. GPU not needed; CPU is fine.

## 1. Install condacolab (RESTARTS the runtime — expected, do not re-run)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. After restart: clone repo + create a Python 3.10 env

In [ ]:
import condacolab; condacolab.check()
!git clone https://github.com/techykajal/streamsense-ott-recsys.git
%cd streamsense-ott-recsys
!conda create -n tfx python=3.10 -y
!conda run -n tfx python --version          # must print Python 3.10.x
!ls requirements-tfx-lock.txt               # the pinned lock must be present in the repo

## 3. Install TFX from the locked file — FAST, no backtracking
Every version is pinned, so pip just downloads and installs (a few minutes of downloads, no
30-minute resolve). This is the fix for the `resolution-too-deep` hang.

In [ ]:
!conda run -n tfx pip install -q -r requirements-tfx-lock.txt
!conda run -n tfx python -c "import tfx, tensorflow as tf; print('TFX', tfx.__version__, '| TF', tf.__version__, 'OK')" 

## 4. Prepare the pipeline's training data
Builds `data/processed/interactions.tfrecord` (the pipeline's input). Content embeddings use the
deterministic fallback (no sentence-transformers needed here).

In [ ]:
!conda run -n tfx python src/features.py

## 5. Run the TFX pipeline END-TO-END  ⭐
ExampleGen → StatisticsGen → SchemaGen → ExampleValidator → Transform → Trainer → Evaluator → Pusher.

In [ ]:
!conda run -n tfx python src/tfx_pipeline.py
print("\n=== Component artifacts produced by the pipeline ===")
!find tfx/ott_ranking -maxdepth 2 -type d 2>/dev/null | sort
print("\n=== Pushed (deployable) model ===")
!ls -R artifacts/ranking_tf 2>/dev/null | head -25 || echo "(Pusher dest = artifacts/ranking_tf in src/tfx_pipeline.py)" 

## 6. Compile the pipeline for Kubeflow (Vertex AI / GKE)

In [ ]:
!conda run -n tfx python src/tfx_pipeline.py --compile-kfp
print("=== ott_pipeline.yaml (first 45 lines) ===")
!head -45 ott_pipeline.yaml

## 7. Done — TFX pipeline executed end-to-end ✅

You ran the full DAG on real x86 Linux + Python 3.10, and compiled the Kubeflow v2 spec.

**Complete production story:**
- **Train (this notebook):** TFX validates data → trains → evaluates → **pushes** a model,
  reproducibly; compiled to a Kubeflow spec for Vertex AI / GKE.
- **Registry:** the pushed model is versioned (POC: `models/`; prod: MLflow / Vertex / S3).
- **Serve (other notebook):** TF Serving / TorchServe / Triton **load** the pushed model.

**Save to your repo:** File → Save a copy in GitHub → `techykajal/streamsense-ott-recsys`,
branch `main`, path `notebooks/StreamSense_Colab_TFX.ipynb`.

**Interview line:** "TFX is my reproducible training pipeline — validate, transform, train,
evaluate, push — compiled to a Kubeflow spec that runs on Vertex AI. Serving is decoupled and just
loads the pushed model. The install was the tricky part: TFX's dependency tree makes pip backtrack,
so I pinned it with a uv-generated lock for a deterministic install."